In [101]:
import os
import uuid
import chromadb
import dotenv

from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import  Document
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from sentence_transformers import  SentenceTransformer
from langchain_openai import ChatOpenAI

In [56]:
def load_all_pdf():
    folder_path = r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset"
    num_docs = 0
    all_docs = []

    for filname in os.listdir(folder_path):
        if filname.lower().endswith(".pdf"):
            # Complete file path
            pdf_path = os.path.join(folder_path, filname)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1
    print("total pdf:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [57]:
all_pdf_documents = load_all_pdf()

total pdf: 1
total pages: 15


In [58]:
# Split Documents

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap= chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [59]:
chunks = split_docs(all_pdf_documents)
len(chunks)

93

In [60]:
# Embedding Manager

class EmbeddingManger:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading Model...", self,model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [61]:
embedding_manger = EmbeddingManger()

Loading Model... <__main__.EmbeddingManger object at 0x000002304F58EF90> all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


In [68]:
# Vector Store( CHORMADB )

class VectorStoreManger:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self.__initialize_store()

    def __initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create a Client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create the Collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}

        )

        print("initalized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # Store => ids, embedding, documents, metadata
        ids = []
        all_metadata = []
        documentss_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_c{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documentss_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(ids=ids, metadatas=all_metadata, documents=documentss_content, embeddings=embeddings_list)

            print("totoal documents added in vector store=", len(documentss_content))
            print("docs in collection:", self.collection.count())

In [69]:
vector_store = VectorStoreManger()

initalized the vector store with collection: pdf
docs in collection: 0


In [70]:
texts = [doc.page_content for doc in chunks]
embeding = embedding_manger.generate_embeddings(texts)
vector_store.add_documents(chunks, embeding)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

embeddings shape: (93, 384)
totoal documents added in vector store= 1
docs in collection: 1
totoal documents added in vector store= 2
docs in collection: 2
totoal documents added in vector store= 3
docs in collection: 3
totoal documents added in vector store= 4
docs in collection: 4
totoal documents added in vector store= 5
docs in collection: 5
totoal documents added in vector store= 6
docs in collection: 6
totoal documents added in vector store= 7
docs in collection: 7
totoal documents added in vector store= 8
docs in collection: 8
totoal documents added in vector store= 9
docs in collection: 9
totoal documents added in vector store= 10
docs in collection: 10
totoal documents added in vector store= 11
docs in collection: 11
totoal documents added in vector store= 12
docs in collection: 12
totoal documents added in vector store= 13
docs in collection: 13
totoal documents added in vector store= 14
docs in collection: 14
totoal documents added in vector store= 15
docs in collection: 15


In [78]:
class RaGRetriver:
    def __init__(self, embedding_manger, vector_store):
        self.embedding_manger = embedding_manger
        self.vector_store = vector_store

    def retrive(self, query, tok_k=5, score_threshold=0.0):
        # Query => Embedding
        query_embeddings = self.embedding_manger.generate_embeddings([query])[0]

        # Sementic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=tok_k
        )
        # cosine similarity
        retreved_dcos = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]
            
            for i, (doc_id, metadata, documents, distances) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distances

                if similarity_score >= score_threshold:
                    retreved_dcos.append({
                        "id": doc_id, 
                        "documents":documents, 
                        "metadata":metadata, 
                        "distance": documents, 
                        "similarty_score": similarity_score, 
                        "rank": i + 1
                    })
            print(f"retrived {len(retreved_dcos)} documents")
        else:
            print("no documents found")
        return retreved_dcos

In [79]:
rag_retriever = RaGRetriver(embedding_manger, vector_store)

In [80]:
rag_retriever.retrive("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrived 5 documents


[{'id': 'doc_cc74e264f-8fa2-4143-a2f1-9d239c50e1fc',
  'documents': 'Decoder: The decoder is also composed of a stack of N = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention\nsub-layer in the decoder stack to prevent positions from attending to subsequent positions. This',
  'metadata': {'content_length': 495,
   'page_label': '3',
   'total_pages': 15,
   'source': 'C:\\Users\\pc\\OneDrive\\Music\\Desktop\\ML_Projects\\Dataset\\1706.03762v7.pdf',
   'author': '',
   'producer': 'pdfTeX-1.40.25',
   'subject': '',
   'doc_index': 20,
   'title': '',
   'creator': 'LaTeX with hyperref',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'page': 2,
   'keywords': '',
   'trapped': '/False',
   'm

In [102]:
# Define api here 

dotenv.load_dotenv()
api_key_openai = os.environ.get("OPENAI_API_KEY")

In [103]:
# Define LLM

llm = ChatOpenAI(
    openai_api_key = api_key_openai,
    model = "gpt-4o", 
    temperature = 0.7, 
    max_tokens = 1024
)


In [104]:
# Generate our retrieval-augmented output

def generate_output(query, retrieval, llm, top_k= 3):
    results = retrieval.retrive(query, top_k)

    context = "\n".join([doc["documents"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context fot the given query")

        # Context + Query
        prompt = f""" use given context to generate the answer for the query 
                     context: {context}
                     Query: {query}"""

        response = llm.invoke(prompt)   # expecting a string as prompt
        return response.content

In [105]:
answer = generate_output("what is encoder-decoder", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrived 3 documents


In [106]:
print(answer)

None
